# Exploratory Data Analysis of the ScreenSpot-Pro Windows Subset

**Name:** John-Ronan Beira  
**Date:** 19 November 2026  
**Dataset:** ScreenSpot-Pro (Li et al., 2025) — 1,020 grounding requests over 17 applications

> **How to follow the notebook.** The analysis is split across four modules instead of being
> pasted into one long sequence of cells. The `%%writefile` cells create those modules in both
> JupyterLab and Google Colab, so the notebook can run with only the libraries listed in the
> setup cell. You can inspect the complete source in the notebook; a test checks that these
> copies match the four files under `src/isdm/eda/`.

## 1. Introduction

### What the data is

ScreenSpot-Pro tests GUI grounding in high-resolution professional software. For each request,
an annotator saw a full-screen Windows screenshot at the machine's native resolution, read an
instruction such as *"Hand-draw cloud line"*, and marked the one rectangular region that
matched it. These are uncropped captures of installed applications, including AutoCAD
Mechanical 2019, Blender 4.0.2, SolidWorks Premium 2018, and Unreal Engine 5.4.4, among others.

The interfaces are dense, the text is small, and some targets occupy only a few hundredths of
a percent of the screen. The authors report that the best model at publication reached 18.9%.
This analysis starts with the geometry: how large are the annotated targets, and how does that
vary across the corpus?

### Where it came from

| Item | Value |
|---|---|
| Source | Hugging Face dataset `likaixin/ScreenSpot-Pro`, revision `main` |
| Link | https://huggingface.co/datasets/likaixin/ScreenSpot-Pro |
| Paper | Li et al. (2025), arXiv:2504.07981 |
| Licence | MIT (dataset card); the screenshots remain subject to the application vendors' rights |
| Files used | 16 `annotations/*_windows.json` files plus `annotations/matlab_macos.json`, and `images/*.png` (1,020 screenshots) |
| Retrieved | 19 November 2026 |
| Local path | `data/screenspot_pro_windows/` |
| Volume | 1,020 requests, 1,020 screenshots, 862.1 MB of PNG |

The release combines 17 per-application annotation files into `grounding_requests.csv`, with
one row per request, and includes every referenced screenshot. Section 2 checks the merged
rows against the source files. Section 3 investigates a platform label that does not match
what the screenshots show.

### Data dictionary

| Column | Meaning | Unit |
|---|---|---|
| `id` | Request identifier, `<application>_<platform>_<n>` (MATLAB rows use `matlab_macos_<n>`) | label |
| `application` | Application the screenshot is taken from | label (17 levels) |
| `group` | Industry category assigned by the benchmark | label (6 levels) |
| `platform` | Platform label in the release; two levels (`windows` 927, `macos` 93) | label |
| `ui_type` | Whether the annotated target is an **icon** or a **text** element | label (2 levels) |
| `instruction` | The English grounding instruction | free text |
| `instruction_cn` | Chinese translation supplied by the dataset | free text |
| `img_filename` | Screenshot path relative to `images/` | label |
| `img_size_w`, `img_size_h` | Screenshot width and height as declared in the annotation | pixels |
| `bbox_x1`, `bbox_y1` | Target box top-left corner, from the top-left screen origin | pixels |
| `bbox_x2`, `bbox_y2` | Target box bottom-right corner | pixels |

The release marks MATLAB as `macos` in `platform`, the request id (`matlab_macos_<n>`) and
the image directory (`matlab_mac/`). The screenshot tells a different story. For example,
`matlab_mac/screenshot_2024-11-30_20-47-44.png` shows a Windows 10 taskbar, right-hand
window controls and a `D:\annotation` working directory. Section 4 corrects the platform
field; the id and image path retain the release's original names.

### What one row represents

Read each row as one grounding request: a screenshot, an instruction and its target box. The
corpus is not a sample of all screen elements, because it labels only the requested target.
It also has repeated image content: section 3 finds 14 pairs of byte-identical files. So the
1,020 requests include 992 byte-distinct images, but that count does not tell us how many
visually distinct screens there are.

### The questions this analysis will answer

1. **Q1 — Target size.** How large is the annotated target relative to its screen, and does
   that differ between icon targets and text targets?
2. **Q2 — Corpus balance.** How evenly are the 1,020 requests spread across the 17 applications
   and 6 industry groups, and what does the imbalance permit?
3. **Q3 — Resolution and size.** Is screen resolution associated with relative target size,
   and can that association be separated from the application each screen comes from?
4. **Q4 — Instruction text.** How long and how repetitive are the instructions, and does that
   vary by target type?

The annotations are enough to answer these four questions. This notebook does not run a
model, so its findings describe the dataset rather than model performance.

## 2. Data loading and first look

Start with the shared setup: it imports the analysis libraries, widens the table display,
sets the plotting theme and fixes the random seed used later for sampling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # use this anywhere you sample

The setup cell does not choose a data path. Jupyter may start the kernel in the notebook
folder or the repository root, so a relative path such as `data/...` could point somewhere
different on each run. The next cell finds the dataset and prepares the folder for the analysis
modules. It walks upward until it finds `data/screenspot_pro_windows`, which lets the same
notebook run from the repository, from `notebooks/`, or in Colab with the data uploaded beside
it. No machine-specific path is stored in the notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

DATASET_FOLDER = Path("data") / "screenspot_pro_windows"
ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / DATASET_FOLDER).is_dir()),
    Path.cwd(),
)
DATA_DIR = ROOT / DATASET_FOLDER

MODULE_DIR = Path("eda_modules")
MODULE_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(MODULE_DIR.resolve()))

print("working directory :", Path.cwd())
print("dataset directory :", DATA_DIR)
print("module directory  :", MODULE_DIR.resolve())

The dataset is in place, and the module folder is ready. The next four cells write the
analysis modules to disk. They are regular Python files with docstrings and type hints; each
cell reports `Overwriting ...` when it runs. You can read the code here before relying on the
results it produces later.

First, `eda_quality` checks the raw data: missing values, duplicates, bounding boxes,
outliers, resolution counts and screenshot integrity.

In [ ]:
%%writefile eda_modules/eda_quality.py
"""Measure data quality: completeness, duplication, validity, and screenshot integrity.

Why this module exists: every claim in an exploratory analysis rests on a measurement of
what is wrong with the data, and those measurements are mechanical. Keeping them here
means the notebook shows *what was measured and what it concluded*, while the counting
logic is importable and testable on its own.

All functions take a DataFrame and return a DataFrame or a plain tuple; none of them plot
or print, so a caller decides how results are displayed.
"""

from __future__ import annotations

import json
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd

# Columns that identify a request rather than measure it.
ID_COLUMNS = ("id", "img_filename")


def requests_vs_annotations(frame: pd.DataFrame, annotation_dir: Path) -> pd.DataFrame:
    """Compare flattened raw annotation records with the merged request table.

    Args:
        frame: merged request table with the 14 source fields used below.
        annotation_dir: directory containing per-application annotation JSON files.

    Returns:
        One row per annotation file with its raw count, exact record matches, unmatched
        raw records, and duplicate raw identifiers. A ``DataFrame.attrs`` entry named
        ``unmatched_merged`` records merged rows absent from the raw annotations.

    Processing:
        1. Flatten each JSON record's ``bbox`` and ``img_size`` arrays into the merged
           table's scalar columns, preserving all source fields in a canonical tuple.
        2. Count those tuples in the merged table, then consume matches while visiting
           sorted annotation files so duplicate rows cannot match the same merged row twice.
        3. Return per-file counts and the number of merged rows left unmatched.

    Side effects: reads JSON files from ``annotation_dir``; does not modify inputs.
    Raises:
        ValueError: a JSON file is not a list of annotation records.
        KeyError: a source record or merged table lacks a required field.
    """
    from collections import Counter

    columns = (
        "id",
        "application",
        "group",
        "platform",
        "ui_type",
        "instruction",
        "instruction_cn",
        "img_filename",
        "img_size_w",
        "img_size_h",
        "bbox_x1",
        "bbox_y1",
        "bbox_x2",
        "bbox_y2",
    )

    def values(record: dict) -> tuple[str, ...]:
        """Flatten a raw JSON record into the merged table's 14-field comparison key."""
        flattened = {key: record[key] for key in columns[:8]}
        flattened.update(
            dict(
                zip(
                    ("img_size_w", "img_size_h"),
                    record["img_size"],
                    strict=True,
                )
            )
        )
        flattened.update(
            dict(
                zip(
                    ("bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"),
                    record["bbox"],
                    strict=True,
                )
            )
        )
        return tuple(str(flattened[column]) for column in columns)

    # 1. Build a multiset, not a set: multiplicity is part of proving records were neither
    #    dropped nor silently duplicated during the merge.
    merged = Counter(frame.loc[:, columns].astype(str).itertuples(index=False, name=None))
    remaining = merged.copy()
    per_file = []

    # 2. Consume exact source-row matches once, retaining duplicate-ID evidence per file.
    for path in sorted(annotation_dir.glob("*.json")):
        records = json.loads(path.read_text())
        if not isinstance(records, list):
            raise ValueError(f"{path} must contain a JSON list of records")
        raw_rows = [values(record) for record in records]
        raw_counts = Counter(raw_rows)
        exact_matches = sum(min(count, remaining[row]) for row, count in raw_counts.items())
        remaining.subtract({row: min(count, remaining[row]) for row, count in raw_counts.items()})
        per_file.append(
            {
                "file": path.name,
                "records": len(raw_rows),
                "exact_matches": exact_matches,
                "unmatched_raw": len(raw_rows) - exact_matches,
                "duplicate_raw_ids": int(pd.Series([r[0] for r in raw_rows]).duplicated().sum()),
            }
        )

    # 3. Leftover multiplicity is evidence of merged rows not represented by the source.
    table = pd.DataFrame(per_file).sort_values("records", ascending=False)
    table.attrs["unmatched_merged"] = sum(remaining.values())
    return table


def missing_table(frame: pd.DataFrame) -> pd.DataFrame:
    """Count and percentage of missing values per column, worst first.

    Args:
        frame: the request table.

    Returns:
        One row per column with ``n_missing`` and ``pct`` columns.

    Percentages matter more than counts at this size: one missing cell in 927 rows is
    0.1%, which is easy to mistake for "no missing data" if only counts are read.
    """
    # 1. Count non-null absences per column.
    table = frame.isna().sum().to_frame("n_missing")
    # 2. Express them as a share of the row count so columns are comparable.
    table["pct"] = (table["n_missing"] / len(frame) * 100).round(3)
    # 3. Sort so the worst column, if any, is read first.
    return table.sort_values("n_missing", ascending=False)


def duplicate_report(frame: pd.DataFrame, text_column: str = "instruction") -> pd.Series:
    """Count exact duplicates of rows, identifiers, and instruction strings.

    Args:
        frame: the request table.
        text_column: the free-text column whose repetition is of interest.

    Returns:
        A named Series of counts.

    The three checks answer different questions. Duplicate rows mean a record was entered
    twice; duplicate ids mean the same request appears twice under one name; repeated
    instruction text is neither — it is a property of the task, recorded here because it
    limits how far the text can disambiguate a target.
    """
    return pd.Series(
        {
            "exact duplicate rows": int(frame.duplicated().sum()),
            "duplicate ids": int(frame["id"].duplicated().sum()),
            "duplicate (screenshot, instruction) pairs": int(
                frame.duplicated(subset=["img_filename", text_column]).sum()
            ),
            "rows sharing an instruction string": int(
                frame[text_column].duplicated(keep=False).sum()
            ),
            "distinct instruction strings reused": int(
                frame.loc[frame[text_column].duplicated(keep=False), text_column].nunique()
            ),
        },
        name="count",
    )


def level_audit(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """List the levels of each categorical column and flag whitespace defects.

    Args:
        frame: the request table.
        columns: categorical columns to audit.

    Returns:
        One row per column with level count, sorted levels, and whether any level carries
        leading or trailing whitespace.

    Two spellings of one category split a group silently: every later share, median and
    test then runs on half the intended rows. The check is cheap, so it runs before any
    grouping rather than after a suspicious result.
    """
    rows = []
    for column in columns:
        values = frame[column].dropna().unique()
        rows.append(
            {
                "column": column,
                "n_levels": len(values),
                "levels": ", ".join(sorted(map(str, values))),
                "no_whitespace": all(str(value) == str(value).strip() for value in values),
            }
        )
    return pd.DataFrame(rows)


def bbox_violations(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Check bounding boxes for impossible or out-of-screen geometry.

    Args:
        frame: table with ``bbox_x1``/``bbox_y1``/``bbox_x2``/``bbox_y2`` and
            ``img_size_w``/``img_size_h``.

    Returns:
        A tuple of (counts per violation type, the offending rows).

    Four independent failures are counted because they have different causes: inverted
    corners mean a coordinate convention error, zero area means an empty label, and an
    out-of-bounds corner means either a wrong coordinate space or an off-by-one at the
    screen edge. Collapsing them into one "invalid boxes" number would hide which repair
    is needed.
    """
    x1, y1 = frame["bbox_x1"], frame["bbox_y1"]
    x2, y2 = frame["bbox_x2"], frame["bbox_y2"]
    width, height = frame["img_size_w"], frame["img_size_h"]

    counts = pd.Series(
        {
            "x2 <= x1 (inverted horizontally)": int((x2 <= x1).sum()),
            "y2 <= y1 (inverted vertically)": int((y2 <= y1).sum()),
            "zero area": int(((x2 - x1) * (y2 - y1) == 0).sum()),
            "x1 < 0": int((x1 < 0).sum()),
            "y1 < 0": int((y1 < 0).sum()),
            "x2 > image width": int((x2 > width).sum()),
            "y2 > image height": int((y2 > height).sum()),
        },
        name="rows affected",
    )

    offending = frame[(x1 < 0) | (y1 < 0) | (x2 > width) | (y2 > height)]
    return counts.to_frame(), offending


def iqr_flags(series: pd.Series, multiplier: float = 1.5) -> tuple[pd.Series, float, float]:
    """Flag values outside ``[Q1 - k*IQR, Q3 + k*IQR]``.

    Args:
        series: values to flag.
        multiplier: fence width in IQRs; 1.5 is the conventional choice.

    Returns:
        A tuple of (boolean mask aligned to ``series``, lower fence, upper fence).

    Flagging is deliberately separate from removing. The mask says "unusual", and only a
    human decision can say "erroneous"; deleting on the flag alone removes exactly the
    real extreme values an analysis of difficulty most needs to keep.
    """
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - multiplier * iqr, q3 + multiplier * iqr
    return (series < lower) | (series > upper), float(lower), float(upper)


def outlier_summary(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Apply the IQR rule to several columns and summarise the result.

    Args:
        frame: the request table.
        columns: numeric columns to test.

    Returns:
        One row per column with the flag count, share of rows, and both fences.
    """
    rows = []
    for column in columns:
        mask, lower, upper = iqr_flags(frame[column])
        rows.append(
            {
                "column": column,
                "n_flagged": int(mask.sum()),
                "pct_flagged": round(mask.mean() * 100, 2),
                "lower_fence": round(lower, 4),
                "upper_fence": round(upper, 4),
            }
        )
    return pd.DataFrame(rows)


def resolution_support(frame: pd.DataFrame) -> pd.DataFrame:
    """Count requests per native screen resolution.

    Args:
        frame: the request table.

    Returns:
        One row per resolution with its request count and share.

    A resolution carried by a handful of rows cannot support a per-resolution claim, so
    the count is the support for any later statement and is read before making one.
    """
    support = frame.groupby(["img_size_w", "img_size_h"]).size().to_frame(name="requests")
    support["share_pct"] = (support["requests"] / len(frame) * 100).round(2)
    return support.sort_index()


def audit_images(frame: pd.DataFrame, image_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Open every referenced screenshot and compare it with its annotation.

    Args:
        frame: the request table.
        image_root: directory that ``img_filename`` values are relative to.

    Returns:
        A tuple of (one audit row per distinct screenshot, the request table joined to it).
        Status ``ok`` means the full image pixel data was decoded, not just its header.

    Processing:
        1. Visit each distinct referenced filename and report missing files immediately.
        2. Open each existing file, force a full pixel decode, and record its dimensions,
           colour mode, and status; convert decode failures into an explicit unreadable row.
        3. Join the per-file audit back to every request that uses that image.

    Side effects: reads image files from ``image_root``; does not modify inputs.
    """
    records = []
    for relative in sorted(frame["img_filename"].unique()):
        path = image_root / relative
        if not path.exists():
            records.append((relative, None, None, None, "missing"))
            continue
        try:
            # PIL loads lazily; explicitly decode pixels so truncated/corrupt payloads fail here.
            from PIL import Image

            with Image.open(path) as image:
                image.load()
                records.append((relative, image.width, image.height, image.mode, "ok"))
        except Exception as error:  # noqa: BLE001 - report the failure, keep auditing
            records.append((relative, None, None, None, f"unreadable: {type(error).__name__}"))

    audit = pd.DataFrame(records, columns=["img_filename", "file_w", "file_h", "mode", "status"])
    joined = frame.merge(audit, on="img_filename", how="left")
    return audit, joined


def size_mismatch(joined: pd.DataFrame) -> pd.DataFrame:
    """Return rows whose real pixel size disagrees with the animated ``img_size``.

    Args:
        joined: output of :func:`audit_images`.

    Returns:
        The offending rows, empty when every file matches its annotation.
    """
    width_differs = joined["file_w"] != joined["img_size_w"]
    height_differs = joined["file_h"] != joined["img_size_h"]
    return joined[width_differs | height_differs]


def average_hash(path: Path, size: int = 8) -> int:
    """Return a 64-bit average hash of an image.

    Args:
        path: image file.
        size: side length of the reduced greyscale thumbnail; 8 yields 64 bits.

    Returns:
        A ``numpy.uint64`` whose bits say whether each thumbnail pixel exceeded the mean.

    The unsigned 64-bit type is load-bearing: a 64-bit hash overflows signed ``int64``, and
    an object-dtype array of Python integers cannot be shifted as a block, which is how the
    pairwise comparison below is vectorised.

    The transform is deliberate: downscaling to 8x8 removes detail, converting to
    greyscale removes colour, and thresholding at the mean removes overall brightness.
    What survives is coarse layout, which is what makes a re-saved or re-captured screen
    hash close to the original while a different screen does not.
    """
    from PIL import Image

    with Image.open(path) as image:
        thumbnail = image.convert("L").resize((size, size), Image.Resampling.LANCZOS)
    pixels = np.asarray(thumbnail, dtype=float).ravel()
    bits = pixels > pixels.mean()

    # Pack the 64 booleans into one integer, most significant bit first.
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return np.uint64(value)


def file_sha256(path: Path) -> str:
    """Return the SHA-256 hex digest of a file's bytes.

    Args:
        path: file to digest.

    Returns:
        Lower-case hex digest.

    Read in chunks rather than whole: the largest screenshots are a few megabytes each and
    there are 927 of them, so streaming keeps the memory cost flat.
    """
    import hashlib

    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(1 << 20):
            digest.update(chunk)
    return digest.hexdigest()


def byte_duplicates(files: Iterable[str], image_root: Path) -> pd.DataFrame:
    """Find files whose bytes are identical.

    Args:
        files: screenshot names relative to ``image_root``.
        image_root: directory holding the screenshots.

    Returns:
        One row per file that shares its digest with another, with the digest and the size
        of its duplicate group; empty when no two files are byte-identical.

    This is the decisive duplicate check. A digest cannot be fooled by two screens looking
    alike — equal digests mean the same file — so it is the check that establishes whether
    the corpus is free of re-saves, while the perceptual hash is only suggestive.
    """
    records = [(name, file_sha256(image_root / name)) for name in files]
    table = pd.DataFrame(records, columns=["img_filename", "sha256"])
    group_sizes = table["sha256"].map(table["sha256"].value_counts())
    duplicated = table[group_sizes > 1].copy()
    duplicated["group_size"] = group_sizes[group_sizes > 1]
    return duplicated.sort_values(["sha256", "img_filename"]).reset_index(drop=True)


def hash_distance_summary(
    files: Iterable[str], hashes: np.ndarray, thresholds: Iterable[int] = (0, 1, 2, 4, 8, 16)
) -> pd.DataFrame:
    """Count screenshot pairs whose perceptual hashes fall within each Hamming threshold.

    Args:
        files: screenshot names, aligned with ``hashes``.
        hashes: average hashes from :func:`average_hash`.
        thresholds: Hamming distances to count pairs at.

    Returns:
        One row per threshold with the pair count, the same-application count, and the
        share of all pairs.

    The same-application column is the diagnostic that matters. An average hash of a
    downscaled screenshot mostly encodes window chrome and panel layout, which every
    capture of one application shares; if all close pairs are within a single application
    and none cross applications, the hash is measuring layout rather than content and
    cannot be used to claim two screens are near-duplicates.
    """
    files = list(files)
    packed = np.asarray(hashes, dtype=np.uint64)
    shifts = np.arange(63, -1, -1, dtype=np.uint64)
    bit_planes = ((packed[:, None] >> shifts) & 1).astype(np.int8)
    distance = (bit_planes[:, None, :] != bit_planes[None, :, :]).sum(axis=2)

    rows, columns = np.triu_indices(len(files), k=1)
    values = distance[rows, columns]
    names = np.asarray(files)
    same_application = np.array(
        [
            names[i].split("/")[0] == names[j].split("/")[0]
            for i, j in zip(rows, columns, strict=True)
        ]
    )

    total_pairs = len(values)
    summary = []
    for threshold in thresholds:
        within = values <= threshold
        summary.append(
            {
                "max_distance": threshold,
                "pairs": int(within.sum()),
                "same_application": int((within & same_application).sum()),
                "cross_application": int((within & ~same_application).sum()),
                "pct_of_all_pairs": round(within.mean() * 100, 3),
                "total_pairs": total_pairs,
            }
        )
    return pd.DataFrame(summary)


Next, `eda_features` creates the measures used throughout the notebook, including relative
target size, position, resolution and text length. It also applies the documented repairs and
returns the cleaning log, keeping each definition in one place.

In [ ]:
%%writefile eda_modules/eda_features.py
"""Derive analysis features and apply the documented cleaning steps.

Why this module exists: "target size" has to mean one thing throughout the analysis. If
each section recomputed it inline, one section could divide by screen area and another by
screen width and the numbers would silently disagree. Deriving the features in one
function makes that impossible.

Cleaning lives beside it because the derived features depend on the cleaned coordinates,
and the log returned with the cleaned frame is the audit trail the report renders.
"""

from __future__ import annotations

import numpy as np
import pandas as pd


def add_derived_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of ``frame`` with the derived analysis columns added.

    Args:
        frame: table with ``bbox_x1``/``bbox_y1``/``bbox_x2``/``bbox_y2``,
            ``img_size_w``/``img_size_h`` and an ``instruction`` column.

    Returns:
        A new DataFrame with geometry, position, resolution and text-length features.

    The relative measures are the point. Absolute pixel area is not comparable between a
    1920x1080 capture and a 5120x1440 capture, so ``target_area_pct`` — the box's share
    of screen area — is the difficulty measure the analysis uses, and the pixel columns
    are kept only as the intermediate values that produce it.
    """
    derived = frame.copy()

    # 1. Box geometry from its corners.
    derived["bbox_w"] = derived["bbox_x2"] - derived["bbox_x1"]
    derived["bbox_h"] = derived["bbox_y2"] - derived["bbox_y1"]
    derived["bbox_area_px"] = derived["bbox_w"] * derived["bbox_h"]
    derived["screen_area_px"] = derived["img_size_w"] * derived["img_size_h"]

    # 2. Relative size. Percent of screen area is the headline measure; the per-axis
    #    shares say whether a target is wide-and-thin or compact.
    derived["target_area_pct"] = derived["bbox_area_px"] / derived["screen_area_px"] * 100
    derived["bbox_w_pct"] = derived["bbox_w"] / derived["img_size_w"] * 100
    derived["bbox_h_pct"] = derived["bbox_h"] / derived["img_size_h"] * 100

    # 3. Screen shape and scale, so resolution effects can be examined separately from
    #    target shape.
    derived["screen_megapixels"] = derived["screen_area_px"] / 1e6
    derived["screen_aspect"] = derived["img_size_w"] / derived["img_size_h"]
    derived["bbox_aspect"] = derived["bbox_w"] / derived["bbox_h"]

    # 4. Where on the screen the target sits, as 0-1 fractions of each axis.
    derived["bbox_cx_rel"] = (derived["bbox_x1"] + derived["bbox_w"] / 2) / derived["img_size_w"]
    derived["bbox_cy_rel"] = (derived["bbox_y1"] + derived["bbox_h"] / 2) / derived["img_size_h"]

    # 5. Distance to the nearest screen edge, scaled by the shorter side so a 4:3 and a
    #    32:9 screen stay comparable. Zero means the target touches a screen boundary.
    edge_gaps = np.column_stack(
        [
            derived["bbox_x1"],
            derived["bbox_y1"],
            derived["img_size_w"] - derived["bbox_x2"],
            derived["img_size_h"] - derived["bbox_y2"],
        ]
    ).min(axis=1)
    derived["edge_distance_pct"] = (
        edge_gaps / derived[["img_size_w", "img_size_h"]].min(axis=1) * 100
    )

    # 6. Instruction length in both units; words are the unit the analysis compares,
    #    characters indicate how long the individual tokens are.
    derived["instruction_chars"] = derived["instruction"].str.len()
    derived["instruction_words"] = derived["instruction"].str.split().str.len()

    return derived


DERIVED_COLUMNS = (
    "target_area_pct",
    "bbox_w_pct",
    "bbox_h_pct",
    "bbox_aspect",
    "screen_megapixels",
    "screen_aspect",
    "bbox_cx_rel",
    "bbox_cy_rel",
    "edge_distance_pct",
    "instruction_chars",
    "instruction_words",
    "bbox_area_px",
    "screen_area_px",
)


def clean_requests(
    frame: pd.DataFrame, derived: pd.DataFrame | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the documented repairs and return the cleaned table with its change log.

    Args:
        frame: the raw request table. Left unmodified.
        derived: optionally a frame already passed through :func:`add_derived_features`;
            when omitted the features are computed here so the returned frame is always
            analysis-ready.

    Returns:
        A tuple of (cleaned frame, cleaning log). The log has one row per change with the
        problem, the action, what it touched, how many rows, and why — and one row
        recording the checks that found nothing, so an empty log cannot be confused with
        a skipped audit.

    Two rules shape the implementation. No row is dropped for a fixable defect: dropping
    removes evidence to repair a coordinate. And every change is counted before it is
    made, so the log states a measurement rather than an impression.
    """
    cleaned = (derived if derived is not None else add_derived_features(frame)).copy()
    entries: list[dict[str, object]] = []

    def record(problem, action, affected, n_rows, reason):
        """Append one log line; a nested helper keeps the call sites readable."""
        entries.append(
            {
                "#": len(entries) + 1,
                "problem": problem,
                "action": action,
                "columns/rows affected": affected,
                "n_rows": n_rows,
                "reason": reason,
            }
        )

    # Change 1: instruction text carries stray whitespace. Strip it and collapse runs of
    # spaces, because the text-length features are derived from this string and a trailing
    # space inflates a character count by one.
    normalized = cleaned["instruction"].str.strip().str.replace(r"\s{2,}", " ", regex=True)
    whitespace_changed = normalized != cleaned["instruction"]
    if whitespace_changed.any():
        affected = int(whitespace_changed.sum())
        cleaned["instruction"] = normalized
        record(
            "instruction text with leading/trailing spaces or doubled spaces",
            "stripped and collapsed whitespace",
            "instruction, 1 column",
            affected,
            "character counts are derived from this string; whitespace is not content",
        )

    # Change 2: a platform label contradicted by the captures themselves. The release
    # files MATLAB as `macos` in the record, the filename and the image directory, and
    # ships no `matlab_windows.json`. Sampled screenshots show a Windows 10 taskbar,
    # right-hand window controls and a `D:\annotation` working directory, and the paper's
    # application table lists MATLAB as Windows, so the label is wrong for these rows.
    mislabelled = cleaned["application"].eq("matlab") & cleaned["platform"].ne("windows")
    if mislabelled.any():
        affected = int(mislabelled.sum())
        cleaned.loc[mislabelled, "platform"] = "windows"
        record(
            f"{affected} MATLAB records labelled platform=macos "
            "(also filename and image directory)",
            "corrected platform to windows",
            "platform, 1 column",
            affected,
            "sampled captures show a Windows 10 taskbar and a D:\\ path; the paper lists MATLAB "
            "as Windows",
        )

    # Change 3: a coordinate outside the screen rectangle. Clamp it to the first valid
    # pixel; the target itself is unambiguous, only the corner is off by one.
    off_screen = cleaned["bbox_y1"] < 0
    if off_screen.any():
        affected = int(off_screen.sum())
        identifiers = ", ".join(cleaned.loc[off_screen, "id"].head(3))
        cleaned.loc[off_screen, "bbox_y1"] = 0
        record(
            f"bbox_y1 negative on {identifiers} (outside the screen)",
            "clamped bbox_y1 to 0",
            "bbox_y1, 1 column",
            affected,
            "minimum valid screen coordinate is 0; target sits flush to the top edge",
        )

    # Nothing else was found: record that explicitly rather than leaving the log with a
    # single line and no statement about the checks that passed.
    record(
        "no missing values, duplicate rows, duplicate ids, inverted or zero-area boxes",
        "no action",
        "none",
        0,
        "measured in the quality assessment; nothing to repair",
    )

    # Any coordinate repaired above changes the derived geometry, so recompute the
    # features rather than leaving a stale width or area beside a corrected corner.
    cleaned = add_derived_features(cleaned.drop(columns=list(DERIVED_COLUMNS), errors="ignore"))

    return cleaned, pd.DataFrame(entries)


def variance_share(frame: pd.DataFrame, value: str, groups: str) -> dict[str, float]:
    """Decompose the variance of ``value`` into a between-group and within-group part.

    Args:
        frame: table containing ``value`` and ``groups``.
        value: numeric column to decompose.
        groups: grouping column.

    Returns:
        ``{"between_pct": ..., "within_pct": ..., "n_groups": ...}``.

    The decomposition answers whether rows inside one group are more alike than rows
    across groups. That decides whether a group-level effect is a real grouping factor or
    just an artefact of pooling. It is computed on the log of the value because a
    right-skewed measure would otherwise have its variance dominated by its largest
    observations, making the between-group share meaningless.
    """
    # 1. Work on a log scale; zero or negative values cannot be logged and indicate a
    #    caller error rather than something to silently drop.
    values = frame[value]
    if (values <= 0).any():
        raise ValueError(f"{value} must be strictly positive to decompose on a log scale")
    logged = np.log10(values)

    # 2. Total sum of squares around the grand mean.
    grand_mean = logged.mean()
    total = ((logged - grand_mean) ** 2).sum()

    # 3. Between-group sum of squares: each group's mean distance from the grand mean,
    #    weighted by its size.
    grouped = logged.groupby(frame[groups])
    between = float(((grouped.mean() - grand_mean) ** 2 * grouped.size()).sum())

    return {
        "between_pct": round(between / total * 100, 2),
        "within_pct": round((1 - between / total) * 100, 2),
        "n_groups": int(frame[groups].nunique()),
    }

`eda_analysis` contains the summaries and tests for distributions, rank comparisons,
correlations, subgroup checks, crosstabs and instruction text. It returns results without
printing them, leaving this notebook to explain what they mean.

In [ ]:
%%writefile eda_modules/eda_analysis.py
"""Compute the summaries and tests the report quotes.

Why this module exists: the analysis repeatedly asks the same three questions — how is
this variable distributed, do these two groups differ, and is this association real or a
composition effect. Each answer is one function, so the notebook can state *which*
question it is answering in Markdown and get a consistent, testable answer underneath.

Every function returns a table or a plain mapping. Nothing prints, so the caller owns the
presentation and the interpretation stays in the report.
"""

from __future__ import annotations

import re
from collections import Counter
from collections.abc import Iterable

import numpy as np
import pandas as pd
from scipy import stats


def numeric_summary(frame: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    """Summarise numeric columns with centre, spread and shape.

    Args:
        frame: the cleaned table.
        columns: numeric columns to summarise.

    Returns:
        One row per column with count, mean, median, standard deviation, IQR, min, max
        and skewness.

    The mean is included deliberately even where it is a poor summary, because the
    contrast with the median is the evidence that the variable is skewed and that the
    median is the right one to report. IQR is appended beside the standard deviation for
    the same reason: it survives the outliers that the standard deviation does not.
    """
    columns = list(columns)
    summary = frame[columns].describe().T
    summary["skew"] = frame[columns].skew()
    summary["IQR"] = frame[columns].quantile(0.75) - frame[columns].quantile(0.25)
    ordered = ["count", "mean", "50%", "std", "IQR", "min", "max", "skew"]
    return summary[ordered].round(4)


def rank_comparison(left: pd.Series, right: pd.Series, left_label: str, right_label: str) -> dict:
    """Compare two groups on a skewed measure with a Mann-Whitney U test.

    Args:
        left: values for the first group.
        right: values for the second group.
        left_label: display name of the first group.
        right_label: display name of the second group.

    Returns:
        A mapping of group sizes, medians, IQRs, the U statistic, its p value, the ratio
        of medians, and the rank-biserial effect size.

    A rank test rather than a t-test because target area spans three orders of magnitude
    and is strongly skewed; the median ratio is reported beside the p value because a
    significant difference on 1,020 requests can be trivially small in practice, and the ratio
    is what says whether the gap matters.
    """
    statistic, p_value = stats.mannwhitneyu(left, right, alternative="two-sided")

    # Rank-biserial correlation, aligned with the U statistic: U counts pairs where the
    # left group exceeds the right, so r = +1 means every left value is *below* every
    # right value and r = 0 means the groups fully interleave.
    effect = 1 - 2 * statistic / (len(left) * len(right))

    return {
        "left_label": left_label,
        "right_label": right_label,
        "left_n": len(left),
        "right_n": len(right),
        "left_median": round(float(left.median()), 6),
        "right_median": round(float(right.median()), 6),
        "left_iqr": (
            round(float(left.quantile(0.25)), 6),
            round(float(left.quantile(0.75)), 6),
        ),
        "right_iqr": (
            round(float(right.quantile(0.25)), 6),
            round(float(right.quantile(0.75)), 6),
        ),
        "u_statistic": float(statistic),
        "p_value": float(p_value),
        "median_ratio": round(float(right.median() / left.median()), 3),
        "rank_biserial": round(float(effect), 3),
    }


def spearman_summary(frame: pd.DataFrame, x: str, y: str) -> dict:
    """Report the Spearman rank correlation between two columns.

    Args:
        frame: the cleaned table.
        x: first column.
        y: second column.

    Returns:
        A mapping with the correlation, its p value, and the number of rows used.

    Spearman rather than Pearson because neither variable is normal and the relation need
    not be linear. The p value is reported but not treated as proof: at this sample size
    almost any non-zero correlation is significant, which is why the correlation itself
    and its subgroup stability carry the argument.
    """
    usable = frame[[x, y]].dropna()
    rho, p_value = stats.spearmanr(usable[x], usable[y])
    return {
        "x": x,
        "y": y,
        "rho": round(float(rho), 3),
        "p_value": float(p_value),
        "n": len(usable),
    }


def subgroup_correlations(
    frame: pd.DataFrame,
    group: str,
    x: str,
    y: str,
    min_rows: int = 20,
    min_levels: int = 3,
) -> pd.DataFrame:
    """Recompute a correlation inside each group to expose a composition effect.

    Args:
        frame: the cleaned table.
        group: grouping column.
        x: first correlated column.
        y: second correlated column.
        min_rows: groups smaller than this are reported as unusable, not tested.
        min_levels: groups whose ``x`` takes fewer distinct values than this cannot
            support a rank correlation and are skipped.

    Returns:
        One row per group with its size, distinct x values, correlation and p value, plus
        a note when the group could not be tested.

    A pooled correlation between two variables that are both determined by group
    membership is not evidence about the variables: it can appear, disappear or reverse
    once group is held constant. Recomputing inside each group is the check, and the
    groups that cannot support one are named rather than quietly dropped.
    """
    rows = []
    for name, chunk in frame.groupby(group):
        distinct = int(chunk[x].nunique())
        if len(chunk) < min_rows or distinct < min_levels:
            rows.append(
                {
                    "group": name,
                    "n": len(chunk),
                    "distinct_x": distinct,
                    "rho": np.nan,
                    "p_value": np.nan,
                    "note": "insufficient size or variation",
                }
            )
            continue
        rho, p_value = stats.spearmanr(chunk[x], chunk[y])
        rows.append(
            {
                "group": name,
                "n": len(chunk),
                "distinct_x": distinct,
                "rho": round(float(rho), 3),
                "p_value": float(p_value),
                "note": "",
            }
        )
    return pd.DataFrame(rows).sort_values("rho")


def crosstab_share(frame: pd.DataFrame, index: str, columns: str) -> tuple[pd.DataFrame, dict]:
    """Build a row-percentage crosstab and test the two variables for independence.

    Args:
        frame: the cleaned table.
        index: row variable.
        columns: column variable.

    Returns:
        A tuple of (row-share percentages with a size column, chi-square result).

    Row percentages rather than raw counts because the row groups differ in size by a
    factor of more than two: raw counts would present the largest group's composition as
    if it were the overall pattern. The chi-square result is returned rather than printed
    so the caller decides whether the test is worth quoting.
    """
    counts = pd.crosstab(frame[index], frame[columns])
    shares = (counts.div(counts.sum(axis=1), axis=0) * 100).round(2)
    shares["n"] = counts.sum(axis=1)

    chi2, p_value, dof, expected = stats.chi2_contingency(counts)
    return shares, {
        "chi2": round(float(chi2), 2),
        "dof": int(dof),
        "p_value": float(p_value),
        "min_expected": round(float(expected.min()), 2),
    }


def application_medians(
    frame: pd.DataFrame, value: str, group: str = "application"
) -> pd.DataFrame:
    """Median and IQR of ``value`` per group, ordered from smallest to largest.

    Args:
        frame: the cleaned table.
        value: numeric column to summarise.
        group: grouping column.

    Returns:
        One row per group with its size, median and quartiles.

    Ordered by median because the comparison being made in the report is "how many times
    larger is the largest group's typical target than the smallest".
    """
    medians = (
        frame.groupby(group)[value]
        .agg(
            n="size",
            median="median",
            q1=lambda series: series.quantile(0.25),
            q3=lambda series: series.quantile(0.75),
        )
        .sort_values("median")
        .round(6)
    )
    return medians


def instruction_summary(frame: pd.DataFrame, text_column: str = "instruction") -> pd.DataFrame:
    """Describe instruction length in words and characters.

    Args:
        frame: the cleaned table.
        text_column: column holding the instruction text.

    Returns:
        One row of length statistics, plus the count of single-word instructions.

    Single-word instructions are counted separately because they are the extreme case for
    disambiguation: a bare "Copy" names a target type and nothing else.
    """
    words = frame["instruction_words"]
    characters = frame["instruction_chars"]
    return pd.DataFrame(
        {
            "words": [words.median(), words.mean(), words.min(), words.max()],
            "characters": [
                characters.median(),
                characters.mean(),
                characters.min(),
                characters.max(),
            ],
        },
        index=["median", "mean", "min", "max"],
    ).round(2).assign(single_word_instructions=int((words == 1).sum()))


def top_terms(
    series: pd.Series,
    stopwords: Iterable[str] = (),
    top_n: int = 20,
    min_length: int = 3,
) -> pd.DataFrame:
    """Count the most frequent content words after stop-word removal.

    Args:
        series: instruction strings.
        stopwords: words to exclude.
        top_n: how many terms to return.
        min_length: shortest token length to keep.

    Returns:
        A DataFrame of ``term`` and ``count``, most frequent first.

    Tokens are lower-cased and reduced to letters and hyphens, because the question is
    which interface concepts the instructions name — not how they were capitalised. A
    stop-word list supplied by the caller keeps this module free of a language assumption
    the data may not meet.
    """
    stopwords = {word.lower() for word in stopwords}
    counter: Counter[str] = Counter()

    for text in series.dropna():
        for token in re.findall(r"[a-z][a-z\-]+", str(text).lower()):
            if token not in stopwords and len(token) >= min_length:
                counter[token] += 1

    return pd.DataFrame(counter.most_common(top_n), columns=["term", "count"])


def position_usage(series: pd.Series, terms: Iterable[str]) -> pd.Series:
    """Mark instructions that name a position or direction.

    Args:
        series: instruction strings.
        terms: spatial terms to look for, matched case-insensitively.

    Returns:
        A boolean Series aligned to ``series``.

    An instruction that says "top left" has already narrowed the search, so a corpus with
    many of them would flatter any model tested on it. Counting them separates genuine
    localisation difficulty from textual spoon-feeding.

    Matching is on word boundaries, which is not cosmetic: a plain substring search counts
    "centerline" (contains "center"), "stop" (contains "top"), "Linear" / "nearby" (contain
    "near"), "corners" (contains "corner") and "desktop" (contains "top") as position words,
    which roughly doubles the result on this corpus.
    """
    pattern = r"\b(?:" + "|".join(re.escape(term) for term in terms) + r")\b"
    return series.str.lower().str.contains(pattern, regex=True)


Finally, `eda_viz` builds the figures. Its functions return `Figure` objects without
rendering them, so labels, log scales and median markers stay consistent across the notebook.

In [ ]:
%%writefile eda_modules/eda_viz.py
"""Build the report's figures.

Why this module exists: a figure is only trustworthy if every one of them is labelled the
same way and uses the same scale for the same quantity. Building them here means the axis
labels, the log scale on skewed measures and the median line are decided once.

Each function returns a :class:`matplotlib.figure.Figure` and draws nothing itself, so the
caller decides when to render it and where it goes. No backend is forced: a notebook
wants its inline backend and a headless run wants Agg, and that is the caller's choice,
not this module's.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Marker colour used for the median line on every distribution figure.
MEDIAN_COLOUR = "darkred"


def plot_distributions(frame: pd.DataFrame, specs: list[tuple[str, str, bool]]) -> plt.Figure:
    """Draw a histogram and a box plot for each requested variable.

    Args:
        frame: the cleaned table.
        specs: one ``(column, axis label, log scale)`` tuple per variable.

    Returns:
        A figure with one row per variable and two columns.

    A histogram alone hides the outlier structure and a box plot alone hides multimodality,
    so both are drawn for the same variable in the same row. A log x-axis is applied where
    the caller asks for it, and the median is marked because on a skewed variable it is the
    summary being reported — a reader can see where it sits rather than take the number on
    trust.
    """
    figure, axes = plt.subplots(len(specs), 2, figsize=(11, 3 * len(specs)))
    axes = np.atleast_2d(axes)

    for row, (column, label, log_scale) in enumerate(specs):
        # Zero cannot be placed on a log axis, and a zero distance means the target is
        # flush with the edge rather than missing; such rows are excluded from the log view.
        values = frame[column].replace(0, np.nan) if log_scale else frame[column]

        sns.histplot(values.dropna(), kde=not log_scale, ax=axes[row, 0], color="steelblue")
        sns.boxplot(x=values.dropna(), ax=axes[row, 1], color="lightsteelblue")

        for axis in axes[row]:
            if log_scale:
                axis.set_xscale("log")
            axis.set_xlabel(label)
        axes[row, 0].set_ylabel("Grounding requests (count)")
        axes[row, 1].set_ylabel("")
        axes[row, 0].set_title(f"Distribution of {column} (n={int(values.notna().sum())})")
        axes[row, 1].set_title(f"Box plot of {column}")
        axes[row, 0].axvline(
            frame[column].median(), color=MEDIAN_COLOUR, linestyle="--", linewidth=1
        )

    figure.tight_layout()
    return figure


def plot_correlation(frame: pd.DataFrame, columns: list[str]) -> plt.Figure:
    """Draw a Spearman correlation heatmap for the requested numeric columns.

    Args:
        frame: the cleaned table.
        columns: numeric columns to correlate.

    Returns:
        A square heatmap figure with annotated coefficients.

    Rank correlation, because the size variables are skewed and the relations need not be
    linear; a Pearson heatmap on these columns would be dominated by a handful of large
    targets. Annotations are printed so a cell can be read without a colour legend, and
    the diverging palette is centred at zero so the sign is visible at a glance.
    """
    correlation = frame[columns].corr(method="spearman")

    figure, axis = plt.subplots(figsize=(8.5, 6.5))
    sns.heatmap(
        correlation,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        cbar_kws={"label": "Spearman rho"},
        ax=axis,
    )
    axis.set_title("Spearman rank correlation, numeric variables")
    axis.set_xlabel("")
    axis.set_ylabel("")
    figure.tight_layout()
    return figure


def plot_categoricals(frame: pd.DataFrame) -> plt.Figure:
    """Draw sorted count bars for application, industry group and target type.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with three panels, one per categorical variable.

    Bars are sorted by count so the imbalance is visible without reading the labels, and
    each bar carries its count or share. A share without its denominator would hide the
    fact that the smallest group holds 45 rows.
    """
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    application_counts = frame["application"].value_counts()
    sns.barplot(
        x=application_counts.to_numpy(), y=application_counts.index, ax=axes[0], color="steelblue"
    )
    axes[0].set_title(f"Requests per application (n={len(frame)})")
    axes[0].set_xlabel("Grounding requests (count)")
    axes[0].set_ylabel("Application")
    for position, value in enumerate(application_counts.to_numpy()):
        axes[0].text(value + 1, position, str(value), va="center", fontsize=8)

    group_counts = frame["group"].value_counts()
    sns.barplot(x=group_counts.index, y=group_counts.to_numpy(), ax=axes[1], color="seagreen")
    axes[1].set_title("Requests per industry group")
    axes[1].set_xlabel("Industry group")
    axes[1].set_ylabel("Grounding requests (count)")
    axes[1].tick_params(axis="x", rotation=30)
    for position, value in enumerate(group_counts.to_numpy()):
        axes[1].text(position, value + 4, str(value), ha="center", fontsize=8)

    type_counts = frame["ui_type"].value_counts()
    sns.barplot(x=type_counts.index, y=type_counts.to_numpy(), ax=axes[2], color="indianred")
    axes[2].set_title("Requests per target type")
    axes[2].set_xlabel("Annotated target type")
    axes[2].set_ylabel("Grounding requests (count)")
    for position, value in enumerate(type_counts.to_numpy()):
        share = value / len(frame) * 100
        axes[2].text(position, value + 4, f"{value} ({share:.1f}%)", ha="center", fontsize=8)

    figure.tight_layout()
    return figure


def plot_area_by_type(frame: pd.DataFrame) -> plt.Figure:
    """Compare target area and target width between icon and text targets.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with a log-scale box plot of relative area and a linear box plot of width.

    Two panels because the two measures answer different questions: relative area compares
    targets across screens of different sizes, while pixel width says how large the label
    physically is. Showing both prevents a resolution difference from being read as a
    target-type difference.
    """
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    sns.boxplot(
        data=frame,
        x="ui_type",
        y="target_area_pct",
        hue="ui_type",
        palette="Set2",
        legend=False,
        showfliers=False,
        ax=axes[0],
    )
    axes[0].set_yscale("log")
    axes[0].set_title("Target area by annotated target type (log scale)")
    axes[0].set_xlabel("Annotated target type")
    axes[0].set_ylabel("Target area (% of screen area, log scale)")

    sns.boxplot(
        data=frame,
        x="ui_type",
        y="bbox_w",
        hue="ui_type",
        palette="Set2",
        legend=False,
        showfliers=False,
        ax=axes[1],
    )
    axes[1].set_title("Target width by annotated target type")
    axes[1].set_xlabel("Annotated target type")
    axes[1].set_ylabel("Target width (pixels)")

    figure.tight_layout()
    return figure


def plot_area_by_application(frame: pd.DataFrame) -> plt.Figure:
    """Draw target area per application, split by target type.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with applications ordered by median target area.

    Ordering by median rather than alphabetically puts the comparison the report makes
    first: the smallest and largest typical targets are at the two ends. Splitting by type
    inside each application shows whether the icon/text gap is consistent or produced by
    one application's composition.
    """
    order = frame.groupby("application")["target_area_pct"].median().sort_values().index

    figure, axis = plt.subplots(figsize=(11, 5.6))
    sns.boxplot(
        data=frame,
        x="application",
        y="target_area_pct",
        order=order,
        hue="ui_type",
        palette="Set2",
        showfliers=False,
        ax=axis,
    )
    axis.set_yscale("log")
    axis.set_title("Target area by application and target type (log scale)")
    axis.set_xlabel("Application (ordered by median target area)")
    axis.set_ylabel("Target area (% of screen area, log scale)")
    axis.tick_params(axis="x", rotation=35)
    axis.legend(title="Target type", bbox_to_anchor=(1.01, 1), loc="upper left")

    figure.tight_layout()
    return figure


def plot_resolution_relationship(frame: pd.DataFrame, medians: pd.DataFrame) -> plt.Figure:
    """Draw the resolution-versus-target-size relation, pooled and per resolution.

    Args:
        frame: the cleaned table.
        medians: one row per resolution with ``screen_label`` and ``target_area_pct``.

    Returns:
        A figure with a pooled scatter and a bar chart of per-resolution medians.

    The pooled panel invites a resolution story and the bar panel tests it: if the pooled
    trend came from a few points, the per-resolution bars will show that the relation is
    not monotone and the claim has to be softened.
    """
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    sns.scatterplot(
        data=frame,
        x="screen_megapixels",
        y="target_area_pct",
        hue="ui_type",
        alpha=0.45,
        s=18,
        palette="Set2",
        ax=axes[0],
    )
    axes[0].set_yscale("log")
    axes[0].set_title("Target area vs screen resolution (pooled)")
    axes[0].set_xlabel("Screen resolution (megapixels)")
    axes[0].set_ylabel("Target area (% of screen area, log scale)")

    sns.barplot(
        data=medians, x="screen_label", y="target_area_pct", ax=axes[1], color="steelblue"
    )
    axes[1].set_title("Median target area by native resolution")
    axes[1].set_xlabel("Native screen resolution (pixels)")
    axes[1].set_ylabel("Median target area (% of screen area)")
    axes[1].tick_params(axis="x", rotation=35)

    figure.tight_layout()
    return figure


def plot_type_share(frame: pd.DataFrame) -> plt.Figure:
    """Draw the icon/text mix within each application as a stacked share bar.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with one horizontal stacked bar per application.

    Row percentages, not counts, because the question is composition: a bar of 82
    PowerPoint requests and a bar of 31 Illustrator requests are only comparable once both
    are scaled to their own total.
    """
    shares = pd.crosstab(frame["application"], frame["ui_type"], normalize="index") * 100
    shares = shares.sort_values("icon")

    figure, axis = plt.subplots(figsize=(11, 5.4))
    shares.plot(
        kind="barh", stacked=True, ax=axis, color=["indianred", "steelblue"], width=0.8
    )
    axis.set_title("Share of icon vs text targets within each application (row percentages)")
    axis.set_xlabel("Share of requests within the application (%)")
    axis.set_ylabel("Application")
    axis.legend(title="Target type", loc="lower right")
    axis.axvline(50, color="black", linestyle=":", linewidth=1)

    figure.tight_layout()
    return figure


def plot_instruction_length(frame: pd.DataFrame) -> plt.Figure:
    """Draw instruction length distributions in words and characters.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with both histograms and the median marked.

    Words and characters are shown together because their ratio is informative: many words
    of few characters each is a different instruction style from few long words, and only
    the pair distinguishes them.
    """
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    sns.histplot(
        frame["instruction_words"], bins=range(1, 18), ax=axes[0], color="steelblue"
    )
    axes[0].set_title("Instruction length in words")
    axes[0].set_xlabel("Words per instruction")
    axes[0].set_ylabel("Grounding requests (count)")
    axes[0].axvline(
        frame["instruction_words"].median(), color=MEDIAN_COLOUR, linestyle="--", linewidth=1
    )

    sns.histplot(frame["instruction_chars"], bins=30, ax=axes[1], color="seagreen")
    axes[1].set_title("Instruction length in characters")
    axes[1].set_xlabel("Characters per instruction")
    axes[1].set_ylabel("Grounding requests (count)")
    axes[1].axvline(
        frame["instruction_chars"].median(), color=MEDIAN_COLOUR, linestyle="--", linewidth=1
    )

    figure.tight_layout()
    return figure


def plot_top_terms(terms: pd.DataFrame) -> plt.Figure:
    """Draw the most frequent instruction terms as a sorted bar chart.

    Args:
        terms: output of :func:`isdm.eda.analysis.top_terms`.

    Returns:
        A horizontal bar figure ordered from least to most frequent.
    """
    figure, axis = plt.subplots(figsize=(8, 5))
    sns.barplot(
        data=terms.sort_values("count"), x="count", y="term", color="steelblue", ax=axis
    )
    axis.set_title("Most frequent instruction terms after stop-word removal")
    axis.set_xlabel("Occurrences (count)")
    axis.set_ylabel("Term")
    figure.tight_layout()
    return figure


def plot_target_crops(
    frame: pd.DataFrame, image_root: Path, applications: list[str], n: int = 2, seed: int = 42
) -> plt.Figure:
    """Crop around the annotated target in sample screenshots and outline the box.

    Args:
        frame: the cleaned table.
        image_root: directory the ``img_filename`` values are relative to.
        applications: applications to sample from, one row of panels each.
        n: screenshots per application.
        seed: sampling seed so the same examples are drawn on every run.

    Returns:
        A figure with one row per application and ``n`` crops per row.

    The crop is the point. A full 5120x1440 screenshot renders a 16-pixel icon as a
    handful of pixels, so the annotated box would be invisible and the check meaningless.
    Zooming to a window around the box makes it possible to see whether the box is on the
    requested element — the one check no summary statistic can perform.
    """
    from PIL import Image

    figure, axes = plt.subplots(
        len(applications), n, figsize=(7 * n, 3.6 * len(applications))
    )
    axes = np.atleast_2d(axes)

    for row, application in enumerate(applications):
        subset = frame[frame["application"] == application].sample(
            min(n, len(frame[frame["application"] == application])), random_state=seed
        )
        for column in range(n):
            axis = axes[row, column]
            if column >= len(subset):
                axis.axis("off")
                continue

            record = subset.iloc[column]
            with Image.open(image_root / record["img_filename"]) as image:
                axis.imshow(image.convert("RGB"))

            axis.add_patch(
                plt.Rectangle(
                    (record["bbox_x1"], record["bbox_y1"]),
                    record["bbox_w"],
                    record["bbox_h"],
                    fill=False,
                    edgecolor="red",
                    linewidth=1.8,
                )
            )

            # Keep a fifth of the screen as context on each side of the target.
            pad_x = record["img_size_w"] * 0.22
            pad_y = record["img_size_h"] * 0.22
            axis.set_xlim(
                max(0, record["bbox_x1"] - pad_x),
                min(record["img_size_w"], record["bbox_x2"] + pad_x),
            )
            axis.set_ylim(
                min(record["img_size_h"], record["bbox_y2"] + pad_y),
                max(0, record["bbox_y1"] - pad_y),
            )
            axis.set_title(
                f"{record['application']} | {record['ui_type']} | {record['instruction'][:32]}",
                fontsize=9,
            )
            axis.set_xlabel("Screen x (pixels)")
            axis.set_ylabel("Screen y (pixels)")

    figure.tight_layout()
    return figure


def plot_pairplot(
    frame: pd.DataFrame,
    columns: list[str],
    hue: str,
    sample_size: int = 500,
    seed: int = 42,
) -> sns.PairGrid:
    """Draw a pairwise grid of several variables coloured by one categorical.

    Args:
        frame: the cleaned table.
        columns: numeric or categorical columns to include, four to six is readable.
        hue: categorical column to colour by.
        sample_size: rows to draw; scatter plots of close to a thousand points become a
            solid mass.
        seed: sampling seed.

    Returns:
        The :class:`seaborn.PairGrid`, whose ``figure`` holds the axes.

    Sampling is a plotting decision only. Every statistic in the report is computed on all
    rows, and the sample size is stated in the figure title so a reader cannot mistake the
    grid for the analysis.
    """
    sample = frame.sample(min(sample_size, len(frame)), random_state=seed)
    grid = sns.pairplot(
        sample[columns],
        hue=hue,
        palette="Set2",
        corner=True,
        plot_kws={"s": 12, "alpha": 0.5},
        diag_kind="kde",
    )
    grid.figure.suptitle(
        f"Pairwise relationships, {len(sample)}-row random sample (seed={seed})", y=1.01
    )
    return grid

Now the notebook imports the four files it just wrote. This checks that the `%%writefile`
paths worked and that the analysis uses its local modules rather than an unrelated package
installed in the environment.

In [ ]:
import eda_analysis as analysis
import eda_features as features
import eda_quality as quality
import eda_viz as viz

from importlib.metadata import version

# Every third-party library this notebook uses, including the two beyond the assignment's
# setup cell: Pillow opens the screenshots, tabulate renders the cleaning log as Markdown.
THIRD_PARTY = ("pandas", "numpy", "matplotlib", "seaborn", "scipy", "pillow", "tabulate")

print("python:", sys.version.split()[0])
for package in THIRD_PARTY:
    print(f"{package:<12}: {version(package)}")
print("modules from:", Path(quality.__file__).parent)

The output above lists the package versions used in this run. They also meet
the assignment's library-listing requirement: seven third-party packages, including Pillow for
opening screenshots and tabulate for rendering the cleaning log. The remaining imports come
from the Python standard library.

Version details matter here. The box plots use seaborn 0.13's `legend=` argument, and
`application_medians` uses named aggregation from pandas 2.2 or later.

When loading the table, the notebook keeps `id` as text because it names a request, not a
quantity. It also reads the four pixel-coordinate columns as integers so they cannot be
mistaken for text later.

In [ ]:
df = pd.read_csv(
    DATA_DIR / "grounding_requests.csv",
    dtype={"id": str, "application": str, "group": str, "platform": str,
           "ui_type": str, "img_filename": str},
)

INTEGER_COLUMNS = ["img_size_w", "img_size_h", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]
df[INTEGER_COLUMNS] = df[INTEGER_COLUMNS].astype("int64")

print("shape (rows, columns):", df.shape)
print("distinct requests    :", df["id"].nunique())
print("distinct filenames   :", df["img_filename"].nunique())
print("applications         :", df["application"].nunique())
print("industry groups      :", df["group"].nunique())

The table has **1,020 rows, 1,020 distinct request ids, 1,020 distinct filenames, 17
applications and 6 industry groups**. Each row and filename maps to one request. That does not
mean every filename contains a different screen; section 3 checks the image bytes.

The first rows all come from AutoCAD because the files were joined alphabetically by
application. The random sample gives a more representative first look at the rows.

In [ ]:
display(df.head(3))

print("random sample:")
display(df.sample(5, random_state=RANDOM_STATE))

As expected, the head contains only AutoCAD; the random sample includes several applications
and resolutions. Its `bbox` coordinates reach into the thousands, alongside image sizes such
as 3840x1080 and 5120x1440. That suggests the boxes use the screenshots' pixel coordinates,
not a normalised 0–1000 scale. Section 3 checks the files directly.

Next, `info()` checks each column's type and non-null count before we calculate summaries.

In [ ]:
df.info()

Every column is populated, `id` remains a string, and the pixel coordinates are integers. The
release has no date field. Section 7 explains why the timestamps embedded in filenames cannot
stand in for one.

The next summary covers all columns at once. Eight of the fourteen are labels, so their means
would not be meaningful; read those columns for their levels and read the numeric columns for
their ranges.

In [ ]:
display(df.describe(include="all").T)

Two details need follow-up. `bbox_y1` reaches **-1**, outside the screen, while the shortest
instruction has only 2 characters. Section 4 repairs the coordinate; the short instructions
remain part of the dataset and limit how much text can distinguish one target from another.

Before analysing the table, compare it with all 17 raw annotation files. Matching row counts
would miss changed fields or repeated records, so the check below compares complete records
and preserves their multiplicities.

In [ ]:
verification = quality.requests_vs_annotations(df, DATA_DIR / "annotations")
display(verification)

print("raw records          :", int(verification["records"].sum()))
print("exact source matches :", int(verification["exact_matches"].sum()))
print("unmatched raw records:", int(verification["unmatched_raw"].sum()))
print("duplicate raw IDs    :", int(verification["duplicate_raw_ids"].sum()))
print("unmatched merged rows:", verification.attrs["unmatched_merged"])


All **1,020** raw records match the merged table across all 14 source fields. The comparison
finds 0 unmatched raw records, 0 unmatched merged rows and 0 duplicate raw ids. That checks
that the merge kept the source data intact, not just that the totals happen to agree.

The totals also match the paper's table: 846 requests across 15 named Windows applications,
93 for MATLAB and 81 in `windows_common`, for 1,020 altogether. The release files MATLAB
under a macOS label (`matlab_macos.json`, `images/matlab_mac/`, `"platform": "macos"`).
Counting only the files labelled Windows would leave out 93, all from MATLAB.
Section 3 checks the screenshots; section 4 records the correction.

## 3. Data quality assessment

This section checks data quality before making repairs. That way, the cleaning log records
measured counts rather than impressions. We check missing values, duplicates, category labels,
bounding boxes, outliers, resolution coverage and the screenshot files.

Start with missingness. The table reports both counts and percentages; for example, one missing
cell among 1,020 rows would be 0.1%, not zero.

In [ ]:
missing = quality.missing_table(df)
display(missing.T)

print("missing cells            :", int(df.isna().sum().sum()))
print("rows with any missing    :", int(df.isna().any(axis=1).sum()))

There are **no missing values**: 0 cells across 0 rows. A missingness-pattern check has
nothing to compare across groups, and every row is available for this analysis. The benchmark
still cannot tell us how often a live capture pipeline fails to produce a usable annotation;
that limitation is noted later.

Next, check duplication at three levels. A repeated row means a record was entered twice; a
repeated id means a request was named twice; a repeated instruction string is neither.

In [ ]:
duplicates = quality.duplicate_report(df)
display(duplicates.to_frame())

print()
reused = df[df["instruction"].duplicated(keep=False)].sort_values("instruction")
print("distinct instruction strings:", df["instruction"].nunique(), "of", len(df), "requests")
display(reused[["id", "application", "ui_type", "instruction"]].head(8))

There are no duplicate rows or ids, so none need to be removed. Instruction strings do repeat: **60
rows share one of 25 repeated strings**, leaving 985 unique instructions out of 1,020. For
example, *"Copy"* appears in both Quartus and AutoCAD, and *"Draw a circle"* appears in
Inventor and SolidWorks. The same words can refer to an icon in one application and a text
label in another.

Keep these rows. Repeated wording is not a data error; it shows that **instruction text alone
cannot identify a target**. That matters when interpreting any later error analysis.

Now check category labels for typos and stray spaces. Either can split one category into two
without causing an error.

In [ ]:
levels = quality.level_audit(df, ("application", "group", "platform", "ui_type"))
display(levels)

print()
print("instructions with leading/trailing whitespace:",
      int((df["instruction"] != df["instruction"].str.strip()).sum()))
print("instructions with double spaces              :",
      int(df["instruction"].str.contains("  ", regex=False).sum()))

The category labels have no stray spaces or casing variants; all 1,020 application names use
the same short form, such as `powerpoint`. The `platform` field is different: it has **two**
levels, 927 `windows` and 93 `macos`, and all 93 `macos` rows belong to MATLAB. That could
mean the subset includes two platforms, or that MATLAB is mislabelled. The screenshot check
will distinguish those possibilities.

Instruction text has a small formatting issue: **5 requests have leading or trailing spaces,
and 4 have a doubled space**. This affects the character counts reported in section 5 before cleaning; section 4 repairs
the spaces and logs the change.

The next check validates box geometry: corners must be ordered, boxes must have non-zero area,
and every box must fit inside its declared screen. These rules help catch coordinates recorded
in the wrong space.

In [ ]:
counts, offending = quality.bbox_violations(df)
display(counts)

print("offending rows:", len(offending))
display(offending[["id", "application", "img_size_w", "img_size_h",
                   "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]])

Only **one row is invalid**: `inventor_windows_60` starts at `y1 = -1` on a 3840x1080 screen.
The box is otherwise a 16x20 icon touching the top edge, so the coordinate is off by one pixel.
No boxes are inverted, empty or larger than their screens. Together, those checks support one
consistent corner convention across the corpus.

Next, the IQR rule flags unusual values for inspection; it does not remove them. Section 6
shows why that distinction matters for this dataset.

In [ ]:
# The outlier rule needs the derived size columns, so build them once here rather than
# recomputing per test. `clean` in section 4 is this frame plus the documented repairs.
geometry = features.add_derived_features(df)

outliers = quality.outlier_summary(geometry, ("bbox_w", "bbox_h", "target_area_pct"))
display(outliers)

flagged, lower, upper = quality.iqr_flags(geometry["target_area_pct"])
print("upper fence (%):", round(upper, 4))
largest = geometry.loc[flagged].nlargest(8, "target_area_pct")
display(largest[["id", "application", "ui_type", "bbox_w", "bbox_h", "target_area_pct"]])

The IQR rule flags **41 target-area rows (4.0% of the corpus)**, 55 rows by width and 149 by
height. The largest box is 2060x119 pixels, covering 4.73% of its screen. It spans part of the
interface rather than a single button; the next largest is 430x284 pixels, or 1.66%.

These are valid targets, not errors. Large regions are easier to locate, so deleting them would
make the benchmark look harder and bias the summaries. We keep them and use medians and
rank-based tests for the heavily skewed size measures.

For resolution, the question is whether each value has enough requests behind it to support a
comparison.

In [ ]:
support = quality.resolution_support(df)
display(support)
print("distinct resolutions:", len(support))

The seven resolutions are not equally represented. **2560x1440 accounts for 430 requests
(42.2%)**, followed by 3840x1080 with 181 (17.8%) and 3840x2160 with 148 (14.5%). The familiar 1920x1080, however, accounts for only **19 requests (1.9%)**, so claims about
1080p screens would rest on a small part of the data. All 93 MATLAB requests use 2560x1440, which contributes to that
resolution's count.

The annotations also declare image dimensions. The next check opens every file to confirm it
exists, fully decodes and matches the declared pixel size.

In [ ]:
audit, joined = quality.audit_images(df, DATA_DIR / "images")
print("status:", audit["status"].value_counts().to_dict())
print("colour modes:", audit["mode"].value_counts().to_dict())
print("distinct screenshots:", audit["img_filename"].nunique())

mismatch = quality.size_mismatch(joined)
print("files whose real size differs from the annotation:", len(mismatch))
display(mismatch[["img_filename", "img_size_w", "img_size_h", "file_w", "file_h"]])

total_bytes = sum(p.stat().st_size for p in (DATA_DIR / "images").rglob("*.png")) / 1e6
print("total image volume:", round(total_bytes, 1), "MB")

All 1,020 files exist and fully decode. They are RGB with no alpha channel, and **all pixel
dimensions match the annotations** (0 mismatches). That matters because a measure such as
"share of screen area" is meaningful only when the box and screenshot use the same pixel
grid.

Now compare image duplication in two ways. A perceptual hash looks for visual similarity; a
SHA-256 digest identifies files with exactly the same bytes. The results differ, so the next
output shows which check is reliable here.

In [ ]:
files = sorted(audit["img_filename"].unique())
digests = quality.byte_duplicates(files, DATA_DIR / "images")
print("files sharing their bytes with another file:", len(digests))
print("distinct duplicate groups:", digests["sha256"].nunique())
display(digests.head(6))

The digest finds **14 exact duplicate pairs**, involving 28 files. That leaves 992 byte-distinct
images, but different bytes do not prove that every image shows a different screen. Unique
filenames did not guarantee unique image contents.

Each pair comes from one application, with filenames indicating capture times 9–36 seconds
apart. The two requests use **different instructions** on the same image. For example,
`vivado_windows_4` asks to "add report in vivado" while `vivado_windows_9` asks to "search
report in vivado". Those are valid separate requests, but the repeated screenshots mean 28
rows are not independent observations.

The next check tests the perceptual hash. If it mistakes shared interface layout for repeated
content, it cannot support a claim that the remaining images are near-duplicates.

In [ ]:
hashes = np.array([quality.average_hash(DATA_DIR / "images" / name) for name in files])
distance_summary = quality.hash_distance_summary(files, hashes)
display(distance_summary)

identical = int(distance_summary.loc[distance_summary["max_distance"] == 0, "pairs"].iloc[0])
print(f"bit-identical hash pairs: {identical} of {int(distance_summary['total_pairs'].iloc[0])}")
print("cross-application pairs within 2 bits:",
      int(distance_summary.loc[distance_summary["max_distance"] == 2, "cross_application"].iloc[0]))

The average hash reports **2,554 pairs with identical hashes** among 519,690 comparisons, compared with 14 exact
duplicate pairs. Of the 3,990 pairs within 2 bits, 3,920 come from the same application and
**70 cross application boundaries**; 30 of those are bit-identical.

That cross-application count shows the problem. An 8x8 greyscale thumbnail captures window
chrome and panel layout, which often look similar within one application. Similar layouts can
also appear in different applications. The hash therefore measures *layout* more than image
content; it cannot even distinguish two different documents open in Blender. We discard it and
use the byte comparison for exact duplicates. On this dataset, the perceptual hash would
create more confusion than useful evidence.

The checks so far found one off-by-one coordinate, no missing values, no duplicate rows or
ids, no corrupt screenshots, 14 pairs of byte-identical images and one platform label that
conflicts with the captures. The perceptual hash also failed to distinguish image content
reliably, so it is not used in the analysis.

## 4. Data cleaning

Clean a copy of the raw table, leaving the source unchanged. Each repair adds an entry to a
log, shown at the end of this section. The function `eda_features.clean_requests` applies the
repairs in the order listed below.

In [ ]:
clean, cleaning_log = features.clean_requests(df)

print("rows before cleaning:", len(df))
print("rows after  cleaning:", len(clean))
print("columns            :", df.shape[1], "->", clean.shape[1])
print()
display(cleaning_log)

The cleaned table still has **1,020 rows**, with 29 columns instead of 14. The three repairs
change labels or coordinates without dropping any requests:

- **93 MATLAB records labelled `macos` are Windows captures.** Sampled screenshots, including
  `matlab_mac/screenshot_2024-11-30_20-47-44.png`, show a Windows 10 taskbar, right-hand
  window controls and a `D:\annotation` working directory. The paper also lists MATLAB as a
  Windows application. The platform label, request id and image folder all carry the same
  incorrect release label.
- **8 instructions contain stray spaces**: 5 have leading or trailing spaces and 4 have a
  doubled space, with one request in both groups. Removing them prevents unexplained changes
  to the character counts in section 5.
- **1 coordinate falls outside the screen** and is clamped to 0. Dropping that row would remove
  one Inventor icon request from the application's 70 rows, or 1.4% of its evidence, to fix a
  one-pixel boundary error.

No rows were dropped. The last log entry also records the checks that found no issue, so the log
shows both what changed and what did not. The table below gives the full record.

In [ ]:
print(cleaning_log.to_markdown(index=False))

The log records all three repairs, their reasons and row counts, plus one no-action entry.
Section 3 contains the checks that led to them. No further cleaning was needed for the analyses that follow.

Next, `add_derived_features` calculates the measures used in the remaining sections. Building
them once keeps "target size" consistent; the summary below checks that the values are
plausible.

In [ ]:
DERIVED = list(features.DERIVED_COLUMNS)
display(clean[DERIVED].describe().T.round(4))

print("target area range (%): ", round(clean["target_area_pct"].min(), 4), "to",
      round(clean["target_area_pct"].max(), 4))
print("median target area (%):", round(clean["target_area_pct"].median(), 4))

`target_area_pct` ranges from **0.002% to 4.73%**, with a median of **0.0382%**.
`edge_distance_pct` runs from 0 for a target touching the screen edge to 49.1, and
`instruction_words` ranges from 1 to 16.

This step does not build a tokenised vocabulary or add an element taxonomy beyond `icon` and
`text`. Choices such as stemming, stop-word removal and additional element classes belong to
modelling; including them here would mix analysis decisions into data preparation.

## 5. Univariate analysis

This section looks at one variable at a time. The summary depends on the variable's type: for
numeric measures, we report centre, spread and shape. Most size variables are right-skewed,
so skewness appears beside the mean and shows whether the mean represents a typical request.

In [ ]:
NUMERIC = ["screen_megapixels", "bbox_w", "bbox_h", "target_area_pct",
           "bbox_w_pct", "bbox_h_pct", "edge_distance_pct",
           "instruction_chars", "instruction_words"]
display(analysis.numeric_summary(clean, NUMERIC))

Every size measure is strongly right-skewed. `target_area_pct` has skewness **20.3**; its mean
of 0.0653% is about 1.7 times its median of 0.0382%. For `bbox_w`, skewness is 6.8, the mean
is 91.6 px and the median is 49 px.

With this much skew, the mean describes a size that few requests have. From here on, **use
the median and IQR** for size summaries and rank-based tests for group comparisons.

`screen_megapixels` is different: its skew is 1.20, but it represents a handful of fixed
resolutions rather than a continuous measurement. The next figure should therefore show spikes.

In [ ]:
SPECS = [
    ("screen_megapixels", "Screen resolution (megapixels)", False),
    ("target_area_pct", "Target area (% of screen area, log scale)", True),
    ("bbox_w_pct", "Target width (% of screen width, log scale)", True),
    ("edge_distance_pct", "Target distance to nearest screen edge (% of short side)", False),
]
viz.plot_distributions(clean, SPECS)
plt.show()

print("targets flush against a screen edge (distance 0):",
      int((clean["edge_distance_pct"] == 0).sum()))

The resolution histogram has the expected spikes for seven fixed configurations. Its largest
peaks are at 4.1 MP (2560x1440) and the 3840-wide resolutions. These data reflect a handful
of capture machines, not a sample of display-panel types.

On a log scale, `target_area_pct` is close to symmetric and roughly log-normally distributed.
Targets range from about 0.002% to 4.7% of screen area. The median, 0.0382%, is about 790
square pixels on a 1920x1080 screen, smaller than a 29x29 button. The long tail in the box plot
comes from the large but valid regions identified earlier.

`edge_distance_pct` has two visible groups: many targets sit within 0–2% of a window border or
screen edge, and 17 sit exactly at the edge. The remaining targets spread across the interior.
Targets at the boundary pose a different grounding problem from small interior targets. Most
boxes span 1% to 6% of screen width.

The next plot counts categories and shows each group size alongside its share, so small groups
are not hidden by percentages alone.

In [ ]:
viz.plot_categoricals(clean)
plt.show()

print(clean["ui_type"].value_counts(normalize=True).mul(100).round(1).to_string())

Illustrator contributes 31, compared with 82 requests in PowerPoint, a 2.6x spread.
The group counts are less balanced: Creative has **297 requests (29.1%)** across 6 applications,
while Dev has **45 (4.4%)** in one, a 6.6x spread. In this corpus, a statement about "professional software"
will lean heavily on Creative; group averages also combine groups of very different sizes.

There are **622 text targets (61.0%)** and **398 icons (39.0%)**. That split is balanced
enough for a type comparison in section 6. By contrast, 12 of the 34 application-by-type
cells contain fewer than 20 requests.

## 6. Bivariate and multivariate analysis

Each comparison pairs a plot with a statistic. We use rank correlations because size is
strongly skewed and the relationships may not be linear. The p-values below are nominal and
unadjusted: requests are clustered within applications, and 14 image pairs are byte-identical.
Treat these tests as descriptive checks, not confirmatory evidence. Inference would need to
account for both sources of dependence.

The correlation matrix is a map of the relationships, not the conclusion. The sections below
follow up on the associations that matter.

In [ ]:
CORRELATED = ["screen_megapixels", "bbox_w", "bbox_h", "target_area_pct",
              "bbox_w_pct", "bbox_h_pct", "edge_distance_pct", "instruction_chars"]
viz.plot_correlation(clean, CORRELATED)
plt.show()

The strongest correlations are built into the measurements: `target_area_pct` depends on
`bbox_w` and `bbox_h`, so their close relationship is not a separate finding. More useful are
the weak negative associations between `screen_megapixels` and size, and the near-zero
relationship between `instruction_chars` and the geometry measures.

Spearman correlation between `bbox_w` and `bbox_w_pct` is 0.96. At these resolutions, they
rank requests almost identically. The analysis still uses relative area because it remains
comparable if higher-resolution screenshots are added later.

Start with Q1: compare target size by type, using group medians, a rank test and an effect size.

In [ ]:
viz.plot_area_by_type(clean)
plt.show()

icon_area = clean.loc[clean["ui_type"] == "icon", "target_area_pct"]
text_area = clean.loc[clean["ui_type"] == "text", "target_area_pct"]
comparison = analysis.rank_comparison(icon_area, text_area, "icon", "text")
for key, value in comparison.items():
    print(f"{key:>16}: {value}")

The median icon occupies **0.0144%** of its screen, compared with **0.0684%** for a text
target, a **4.7x** difference. The Mann–Whitney result is U = 29,449, nominal unadjusted
p = 7.2e-94, with rank-biserial **+0.762**. Under this sign convention, icon targets rank
almost entirely below text targets. The size gap is descriptive; the row-level p-value does not
account for clustering.

Text boxes are also wider and thinner: their median aspect ratio is 3.62, compared with 1.05
for icons. The right-hand panel shows that width difference.

Comparing target size by type makes the difference clear. Icons make up roughly 39% of the corpus and have a median area 4.7x smaller than text targets. A pooled size figure would obscure that gap.

Q2 turns to the application mix: how much do target sizes vary across applications, and what
part of that difference comes from target type?

In [ ]:
app_medians = analysis.application_medians(clean, "target_area_pct")
display(app_medians)

smallest, largest_app = app_medians.index[0], app_medians.index[-1]
print(f"smallest median: {smallest} = {app_medians['median'].iloc[0]:.4f}%")
print(f"largest  median: {largest_app} = {app_medians['median'].iloc[-1]:.4f}%")
print(f"ratio           = {app_medians['median'].iloc[-1] / app_medians['median'].iloc[0]:.1f}x")

viz.plot_area_by_application(clean)
plt.show()

Median target size varies **11.8x** across applications, from **0.0067%** of the screen in
Origin to **0.0783%** in MATLAB. The plot suggests two reasons: interface style and the mix of
target types. Origin and Stata requests often point to small toolbar icons. SolidWorks requests
include feature-manager rows, while MATLAB targets are mostly text (79.6%) and include larger
menu or panel labels. Within applications that have both types, text targets usually remain
larger than icons, so one unusual application does not explain the Q1 gap.

Q3 asks whether screen resolution is associated with relative target size. The pooled
correlation is only the first step; we also need to see whether resolution can be separated
from application.

In [ ]:
res_medians = (clean.groupby(["img_size_w", "img_size_h", "screen_megapixels"])
               ["target_area_pct"].median().reset_index())
res_medians["screen_label"] = (res_medians["img_size_w"].astype(str) + "x"
                               + res_medians["img_size_h"].astype(str))

viz.plot_resolution_relationship(clean, res_medians)
plt.show()

display(res_medians[["screen_label", "screen_megapixels", "target_area_pct"]].round(4))
print(analysis.spearman_summary(clean, "screen_megapixels", "target_area_pct"))

Across the full dataset, higher resolution is weakly associated with smaller relative targets:
Spearman rho = **-0.088** (nominal unadjusted p = 0.0048). The medians do not follow a steady
trend. For example, 3840x2160 has a median of 0.0153%, 3840x1080 has 0.0577%, and the 19
requests at 1920x1080 have 0.0278%.

A rho near -0.09 is a small association, even though the nominal unadjusted p-value is 0.005
at n = 1,020. Before interpreting it, check within each application: application choice could
be linked to both resolution and target size.

In [ ]:
subgroups = analysis.subgroup_correlations(
    clean, "application", "screen_megapixels", "target_area_pct")
display(subgroups)

varied = int((subgroups["distinct_x"] > 1).sum())
print("applications with more than one native resolution:", varied, "of", len(subgroups))
print("applications that can support a within-group correlation:",
      int(subgroups["rho"].notna().sum()))

The subgroup check reveals a design limit, not a smaller within-group effect. Fifteen of the
seventeen applications were captured at exactly **one** native resolution. Quartus and
Windows Common have two; all 93 MATLAB requests use 2560x1440. Resolution is nearly
constant within each application, so **this subset cannot separate resolution from
application**.

That is why Q3 has no within-application answer. The pooled rho of -0.088 compares
applications with different resolutions; it cannot tell us whether a larger screen makes
targets relatively smaller. To test that, the same application would need captures at several
resolutions. This benchmark has no such comparison, though the project's test matrix could
provide one.

Before leaving Q2, check target-type mix by application. It helps explain why application
averages can be hard to compare.

In [ ]:
shares, chi_square = analysis.crosstab_share(clean, "application", "ui_type")
display(shares.sort_values("icon", ascending=False))

print()
for key, value in chi_square.items():
    print(f"{key:>14}: {value}")

viz.plot_type_share(clean)
plt.show()

The mix of target types differs across applications (chi-square = **192.9**, dof = 16, nominal
unadjusted p = 2.2e-32). Every expected cell exceeds 12, meeting the Pearson test's expected-count
condition. The independence assumption still does not hold under application clustering. Stata
has **83.7%** icon targets and EViews 14.0%; Quartus has 71.1%, Illustrator 61.3%, and Inventor,
SolidWorks and Unreal Engine sit near 16–18%. MATLAB has 20.4%.

This mix helps explain the application-level differences. Comparing Stata with Illustrator
means comparing many toolbar icons with many text labels. Read each application's median
alongside its target-type mix; section 6 reports the size gap by type for that reason.

The final question looks at instruction text and target type.

In [ ]:
viz.plot_instruction_length(clean)
plt.show()

display(analysis.instruction_summary(clean))
print(analysis.spearman_summary(clean, "instruction_words", "target_area_pct"))

icon_words = clean.loc[clean["ui_type"] == "icon", "instruction_words"]
text_words = clean.loc[clean["ui_type"] == "text", "instruction_words"]
words_by_type = analysis.rank_comparison(icon_words, text_words, "icon", "text")
print()
print("instruction words, icon vs text:", words_by_type["left_median"], "vs",
      words_by_type["right_median"], "| p =", round(words_by_type["p_value"], 4))

Instructions are brief: the median is **4 words and 20 characters**, and **78 requests (7.6%)**
use a single word. Most contain 3–5 words.

Length does not meaningfully separate target types (median 4 words in both groups, nominal
unadjusted p = 0.45, rank-biserial -0.03). It is also unrelated to target size (rho = -0.023,
nominal unadjusted p = 0.46). These negative results matter: longer instructions do not signal
a harder target, so length is not a useful difficulty feature in this dataset.

A pair grid provides one final view of the relationships, using a sample small enough to read.

In [ ]:
PAIR_COLUMNS = ["target_area_pct", "bbox_w_pct", "edge_distance_pct",
                "instruction_words", "ui_type"]
viz.plot_pairplot(clean, PAIR_COLUMNS, hue="ui_type", sample_size=500, seed=RANDOM_STATE)
plt.show()

The pair grid shows the same two patterns: text targets tend to have larger `target_area_pct`
and `bbox_w_pct` across instruction lengths, while instruction length itself separates
nothing. Icon targets form a tighter, smaller cluster.

The plot uses a 500-row sample so the points remain visible. All statistics reported above use
all 1,020 rows.

## 7. Type-specific analysis

The assignment lists eleven data types to check. The table below marks which ones this dataset
supports and gives a reason for each exclusion.

| Row in the assignment's table | Applies? | Reason |
|---|---|---|
| Dates or timestamps | **No** | No capture timestamp in the release; explained below rather than improvised. |
| A target variable | **Yes** | `ui_type` and the geometry features are candidate predictors; balance checked above. |
| Groups or repeated measures | **Yes** | Requests are nested in 17 applications and 6 groups. |
| Free text | **Yes** | 1,020 instructions; length, vocabulary and repetition analysed below. |
| Images | **Yes** | 1,020 PNGs; existence, size, mode, corruption and duplication checked. |
| Audio or signals | No | No audio data. |
| Geographic data | No | No coordinates. |
| Survey or Likert data | No | No respondent-level items; `instruction_cn` is a translation, not a second respondent. |
| More than 30 columns | No | 29 columns after derivation, 9 analysed as numeric. |
| More than 1,000,000 rows | No | 1,020 rows; no sampling of summaries needed. |
| Several linked tables | Partly | 17 annotation files plus an image tree, joined on `img_filename`; verified in section 2. |

### Images

First compare image counts and formats across applications, then inspect the annotated boxes
on the screenshots. The measurements can test consistency; only the image can show whether a
box actually surrounds the requested target.

In [ ]:
image_table = (clean.groupby("application")
               .agg(screenshots=("img_filename", "nunique"),
                    megapixels_median=("screen_megapixels", "median"),
                    width_min=("img_size_w", "min"),
                    width_max=("img_size_w", "max"),
                    aspect_median=("screen_aspect", "median"))
               .sort_values("screenshots", ascending=False))
image_table["shape"] = image_table["aspect_median"].round(2).map(
    {1.33: "4:3", 1.5: "3:2", 1.6: "16:10", 1.78: "16:9", 3.56: "32:9"}).fillna("other")
display(image_table.round(3))

print("screenshots:", clean["img_filename"].nunique(),
      "| distinct contents:", clean["img_filename"].nunique() - len(digests))
print("colour modes:", audit["mode"].unique().tolist(),
      "| unreadable:", int((audit["status"] != "ok").sum()))

All 1,020 screenshots are intact RGB PNGs. Most use 16:9 or 32:9 dimensions; a few are 16:10
or 3:2. Each request has its own file, so image counts match request counts, ranging from 31
in Illustrator to 93 in MATLAB.

The six crops below show three applications with different interface densities. Each includes
the annotated box. Check whether it lands on the requested element; no summary statistic can
replace that visual inspection.

In [ ]:
viz.plot_target_crops(clean, DATA_DIR / "images",
                      ["origin", "windows_common", "solidworks"], n=2, seed=RANDOM_STATE)
plt.show()

The red box lands on the requested element in all six crops: an Origin toolbar icon, a Windows
system entry and a SolidWorks feature-manager row.

The crops also make the targets large enough to inspect. At 5120x1440, a 16-pixel icon occupies
only a few pixels on the full screen. Each crop keeps roughly a fifth of the screen as context
while zooming in on the target.

### Free text

The instructions are short. The next checks look at their length, vocabulary and repetition
to see how much they help distinguish one target from another.

In [ ]:
STOPWORDS = {"the", "a", "an", "to", "of", "in", "on", "for", "with", "and", "or", "at", "by",
             "from", "this", "that", "it", "is", "as", "into", "all"}
terms = analysis.top_terms(clean["instruction"], STOPWORDS, top_n=20)
display(terms)

viz.plot_top_terms(terms)
plt.show()

The most common words are interface verbs and nouns: *open* (71), *select* (66), *change*
(58), *tool* (50), *add* (47), *file* (42), *plot* (41) and *edit* (39). Other frequent terms
include *view*, *create*, *insert*, *close*, *selected*, *color* and *set*.

A product name also appears often: `vivado` occurs 70 times, second only to *open*. Some
requests say things like "open exist project in vivado" or "search report in vivado". Instructions therefore partly identify the application. In this dataset, application and screen resolution cannot be separated.

Many instructions name a target type but not its location. The next check counts how often
they give a position or direction.

In [ ]:
POSITION_TERMS = ["top", "bottom", "left", "right", "upper", "lower", "middle", "center",
                  "centre", "corner", "side", "next to", "above", "below", "near", "inside"]
mentions_position = analysis.position_usage(clean["instruction"], POSITION_TERMS)

print("instructions naming a position:",
      int(mentions_position.sum()),
      f"({mentions_position.mean() * 100:.1f}%)")
display(pd.crosstab(mentions_position.map({True: "mentions position", False: "no position"}),
                    clean["ui_type"], normalize="index").mul(100).round(1))

print()
display(clean.loc[mentions_position, ["instruction", "ui_type", "target_area_pct"]].head(6))

Only **25 instructions (2.5%)** give a position or direction: 13 icon requests (3.3% of the icon
group) and 12 text requests (1.9%). The examples include *"Top left corner"* and *"align the
text to the center"*. With just 25 of 1,020 requests, these instructions are too few to change
the overall size findings in section 6.

A simple substring search returns 37 rows. It overcounts because it also matches words such as
*"centerline"*, *"stop"*, *"Linear"*, *"nearby"*, *"corners"* and *"desktop"*, even when they do
not describe a position. Matching complete words gives 25 rather than 37.

The instructions usually say what to find, while the screenshot shows where it is. Along with
the 25 strings reused across 60 requests, this means **instruction text alone cannot identify
a target** or serve as a join key in a later failure analysis.

### Groups and repeated measures

Requests are grouped by application. The next question is whether different rows can also
refer to the same screenshot. Section 3 found 14 duplicate pairs; the variance decomposition
below measures how much target-size variation lies between applications and how much within them.

In [ ]:
per_screen = clean["img_filename"].value_counts()
print("requests per filename: max =", per_screen.max(),
      "| filenames with more than one request:", int((per_screen > 1).sum()))

display(clean.groupby("group").agg(applications=("application", "nunique"),
                                   requests=("id", "size"))
        .sort_values("requests", ascending=False))

share = features.variance_share(clean, "target_area_pct", "application")
for key, value in share.items():
    print(f"{key:>12}: {value}")

Each filename has one request, but 14 pairs of files contain the same image (section 3).
Those 28 requests are therefore dependent, even though they have different instructions. The
limitations section returns to this distinction.

**16.9% of the variance in log target area is between applications**, with 83.1% within them.
Target sizes vary substantially inside applications, while application membership still
accounts for a measurable share of the variation.

### Time

The release has **no capture date, session id or ordering variable**, so a time analysis is not
possible. Filenames such as `screenshot_2024-11-06_19-36-34.png` contain an apparent
timestamp, but the number of capture sessions and machines is unknown. The 14 duplicate pairs
are only 9–36 seconds apart, which makes the filename stamp a capture clock, not a study
timeline.

A trend built from those strings would invent information the release does not contain. To
study changes across sessions or capture days, we would need the original capture metadata.

## 8. Summary of findings

1. The corpus holds **1,020 grounding requests across 17 applications and 6 industry groups**,
   matched exactly across all 14 source fields against the 17 raw annotation files (0 unmatched
   source rows, 0 unmatched merged rows). The files contain 14 pairs of byte-identical screenshots,
   captured 9–36 seconds apart, each carrying two different instructions; visual uniqueness of
   the remaining byte-distinct files is not established.
2. **MATLAB is a Windows capture mislabelled `macos`** in the release. Correcting the platform label expands the Windows-labelled subset from 927 to 1,020 rows;
   the release total remains 1,020. The evidence for that classification is the captures themselves.
3. The median annotated target covers **0.0382% of its screen** (IQR 0.014–0.081), about 791 px² on
   a 1920x1080 display, with a range from 0.002% to 4.73%.
4. Icon targets are **4.7x smaller in median relative area than text targets** (0.0144% vs 0.0684%,
   Mann-Whitney U = 29,449, nominal unadjusted p = 7.2e-94, rank-biserial +0.76), the largest observed size difference between target types.
5. Median target size varies **11.8x across applications**, from 0.0067% (Origin) to 0.0783%
   (MATLAB), partly interface style and partly target-type mix.
6. Target-type mix differs sharply by application (**chi-square = 192.9, dof = 16, nominal unadjusted p = 2.2e-32**;
   Stata 83.7% icon against EViews 14.0%), so application averages are not comparable at face value.
   **16.9% of the variance in log target area is between applications**, 83.1% within them.
7. Screen resolution **cannot be separated from application** here: 15 of 17 applications were
   captured at a single resolution, and the pooled association is weak (rho = -0.088, nominal unadjusted p = 0.005).
8. Instructions have a median length of **4 words and 20 characters**; 7.6% use a single word.
   Length is unrelated to target size (rho = -0.023, nominal unadjusted p = 0.46) or target type
   (nominal unadjusted p = 0.45).
9. Only **2.5% of instructions name a position** (25 requests), and 25 instruction strings are
   reused across 60 requests, so the text cannot disambiguate a target on its own.
10. Data quality is high but not perfect: **no missing values, no duplicate rows or ids, no corrupt
    files after full pixel decoding**, one bounding-box coordinate at -1 and 8 instructions with stray whitespace (both
    repaired), 14 pairs of byte-identical screenshots, and one mislabelled platform.

### Answers to the four questions

**Q1 — Target size.** The median target covers 0.0382% of screen area, and sizes are roughly
log-normal across more than three orders of magnitude. Icon targets have a 4.7x smaller median
area than text targets, a gap seen in nearly every application containing both types. Report
the two types separately; one pooled figure would average distinct populations.

**Q2 — Corpus balance.** The 1,020 requests cover 17 applications (31–93 each) and 6 groups
(45–297 each). Counts vary more sharply by group: Creative contributes 29.1% of the corpus,
while Dev contributes 4.4%. Read group averages with their sample sizes, and application
averages with target-type mix (chi-square = 192.9, nominal unadjusted p = 2.2e-32).

**Q3 — Resolution and size.** This dataset **cannot answer the question**. In the pooled data,
higher resolution is weakly associated with smaller relative targets (rho = -0.088). But 15 of
17 applications were captured at only one resolution, leaving no within-application comparison.
The pooled association could reflect application differences instead. To separate the two, the
same application needs to be captured at several resolutions.

**Q4 — Instruction text.** Instructions are short (median 4 words and 20 characters) and use
similar styles across target types. Interface words dominate, and `vivado` appears in 70
instructions. Length is unrelated to target size or type. Only 25 instructions (2.5%) give a
position, and 60 requests reuse an instruction string. The text usually says *what* to find,
not where to find it.

**A question this data cannot answer:** whether any geometry feature *predicts* or *causes*
grounding failure. The release has no model outputs, success labels or confidence scores. The
findings describe the corpus, not model performance.

## 9. Limitations and next steps

### Limitations

- Each screenshot labels one requested element. The other elements remain unannotated, so this
  dataset cannot evaluate element detection.
- The release has no grounding outcomes, success labels or confidence scores. The size measures
  describe the task, not any model's accuracy.
- The images repeat: 28 requests use 14 pairs of byte-identical screenshots. Their instructions
  differ, so both requests are valid, but they are not independent observations. Requests are
  also grouped within 17 applications. The row-level p-values are nominal and unadjusted;
  inferential tests need to account for both kinds of dependence.
- Resolution cannot be separated from application here. Fifteen of 17 applications have only
  one resolution, leaving Q3 unanswered. This matters for any later model that uses resolution
  as a feature.
- Capture metadata is missing. DPI and display scaling, theme, zoom and application build were
  not recorded, so the application differences in section 6 cannot be attributed to interface
  design rather than rendering conditions.
- Expert annotators selected these 1,020 requests to be representatively *hard*. Their median
  target size does not estimate target sizes in professional software generally, and the corpus
  supports no prevalence claim.
- The 6 industry groups come from the benchmark, and the smallest contains 45 rows. Treat group
  averages as limited evidence.
- The 41 IQR-flagged target-area values were inspected and kept as legitimate large regions.
  Medians and rank statistics remain appropriate for the skewed distribution.
- There is no time dimension: the release has no timestamp, so it cannot show drift, session
  effects or recency.
- Exact duplicates number 14, but the perceptual hash reports 2,554 bit-identical pairs,
  including 30 collisions across applications. Use content digests or pixel-level comparisons
  for later deduplication.

### Next steps

- Add outcome labels and test whether relative target area and edge distance predict failure.
  Label each request `correct`, `mislocalized`, `not_found` or `ambiguous`, using the unit in the
  project's analysis-unit decision.
- Keep target types separate in later models, or include their interaction. Their median areas
  differ by 4.7x, so a single coefficient across icons and text would be misleading.
- Capture the same application at several resolutions. That controlled comparison would separate
  resolution effects from application effects, which Q3 cannot do with this benchmark.
- Account for the 14 duplicate screens whenever a test assumes independent requests. Group the
  repeated images or keep one request per screen.
- Check platform labels against screenshots in the project's own pipeline. The release labels
  the Windows captures as macOS in three places, so metadata alone is not enough.
- Use content digests rather than perceptual hashes for screenshot deduplication, then check
  whether the project's pipeline produces the same one-target-per-screen format. On this corpus,
  the average hash collides more often than it distinguishes images.
- Compare this benchmark with self-collected screenshots to see whether its target-size profile
  transfers to the project's target environment.

## 10. References

- Li, K., Meng, Z., Lin, H., Luo, Z., Tian, Y., Ma, J., Huang, Z., & Chua, T.-S. (2025).
  *ScreenSpot-Pro: GUI Grounding for Professional High-Resolution Computer Use.*
  arXiv:2504.07981. https://doi.org/10.48550/arXiv.2504.07981
- Dataset: `likaixin/ScreenSpot-Pro`, Hugging Face. https://huggingface.co/datasets/likaixin/ScreenSpot-Pro
  — retrieved 19 November 2026, revision `main`; MIT licence covering the annotations.
- Benchmark page and leaderboard: https://gui-agent.github.io/grounding-leaderboard
- pandas, NumPy, Matplotlib, seaborn, SciPy and Pillow — library documentation, used for the
  analysis and the figures.
- AI assistance: an AI coding assistant (Hermes Agent) was used to draft the analysis modules and
  the download script `src/isdm/data/fetch.py`. Every figure and number in this notebook is
  produced by the executed cells above and was checked against the raw annotation files.

## Appendix: exporting the executed notebook to PDF

This is an export utility, not part of the analysis. Run **Kernel → Restart & Run All**, save the
notebook (`Ctrl+S`), then run the cell below. It writes:

- `runs/eda/EDA_Beira_John-Ronan.pdf` using `nbconvert` → LaTeX → `xelatex`;
- `runs/eda/EDA_Beira_John-Ronan.html` as the assignment's accepted fallback, which needs no LaTeX.

PDF export needs `xelatex` and `pandoc`. This repository's dev dependencies provide pandoc through
`pypandoc-binary`; `xelatex` comes from a TeX distribution. If either is missing, the cell prints
the requirement and still writes the HTML. The notebook itself remains runnable without them.


In [ ]:
# Export the saved notebook to PDF (LaTeX) and HTML (fallback).
# Run All first, save the notebook, then run this cell. This is a utility, not analysis.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

NOTEBOOK_NAME = "EDA_Beira_John-Ronan.ipynb"

# Locate the notebook whether the kernel starts in the repo root, notebooks/, or Colab.
search = [Path.cwd(), *Path.cwd().parents]
notebook = next(
    (p for p in (root / "notebooks" / NOTEBOOK_NAME for root in search) if p.is_file()),
    None,
) or next(
    (root / NOTEBOOK_NAME for root in search if (root / NOTEBOOK_NAME).is_file()),
    None,
)
if notebook is None:
    raise FileNotFoundError(f"Could not find {NOTEBOOK_NAME}")

repo_root = notebook.parent.parent if notebook.parent.name == "notebooks" else notebook.parent
out_dir = repo_root / "runs" / "eda"
out_dir.mkdir(parents=True, exist_ok=True)

saved = json.loads(notebook.read_text())
if not any(cell.get("outputs") for cell in saved["cells"] if cell.get("cell_type") == "code"):
    print("WARNING: the saved notebook has no outputs. Run All and save before exporting.")

python = sys.executable
env = os.environ.copy()

# nbconvert's LaTeX path needs pandoc. Prefer one on PATH, else pypandoc's bundled binary.
pandoc = shutil.which("pandoc")
if pandoc is None:
    try:
        import pypandoc
        pandoc = pypandoc.get_pandoc_path()
    except Exception:
        pandoc = None
if pandoc:
    env["PATH"] = str(Path(pandoc).parent) + os.pathsep + env.get("PATH", "")

def run(command, cwd=None):
    return subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)

# HTML always works and is the assignment's accepted fallback.
html = run([python, "-m", "nbconvert", "--to", "html",
            "--output-dir", str(out_dir), "--output", "EDA_Beira_John-Ronan.html", str(notebook)])
html_path = out_dir / "EDA_Beira_John-Ronan.html"
print("HTML:", html_path if html.returncode == 0 else html.stderr[-1200:])

xelatex = shutil.which("xelatex", path=env["PATH"])
if pandoc is None or xelatex is None:
    print(
        "PDF skipped: install pandoc and xelatex, then re-run. "
        "The HTML export above is complete."
    )
else:
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        latex = run([python, "-m", "nbconvert", "--to", "latex",
                     "--output-dir", str(tmp), "--output", "notebook.tex", str(notebook)])
        tex = tmp / "notebook.tex"
        if latex.returncode or not tex.is_file():
            print("LaTeX export failed:", latex.stderr[-1200:])
        else:
            # pandoc 3.x emits \LTcaptype{none}; the nbconvert template needs this counter.
            source = tex.read_text()
            if r"\newcounter{none}" not in source:
                source = source.replace(
                    r"\begin{document}", r"\newcounter{none}" + "\n" + r"\begin{document}", 1
                )
                tex.write_text(source)
            for _ in range(2):
                built = run(
                    [xelatex, "-interaction=nonstopmode", "-halt-on-error", tex.name],
                    cwd=tmp,
                )
            pdf = tmp / "notebook.pdf"
            if built.returncode == 0 and pdf.is_file():
                target = out_dir / "EDA_Beira_John-Ronan.pdf"
                shutil.copyfile(pdf, target)
                print("PDF:", target)
            else:
                print("xelatex failed:", built.stderr[-1200:])
